In [1]:
from pathlib import Path
import json
import sys

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()),
    Path.cwd()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.audio_preprocessing import preprocess_audio_for_transcription
from src.transcription import transcribe_audio_file
from src.utils import load_environment, validate_environment

# Load and validate environment variables
load_environment()

missing_keys = validate_environment()
if missing_keys:
    raise EnvironmentError(
        f"Missing required environment variables: {', '.join(missing_keys)}"
    )

# Original podcast file
audio_path = repo_root / "sources" / "The_Blueprint_For_Trustworthy_AI.m4a"

# Preprocess audio to Whisper-compatible size
preprocessed_audio_path = preprocess_audio_for_transcription(audio_path)

print(f"Original file: {audio_path}")
print(f"Preprocessed file: {preprocessed_audio_path}")

print(f"Original size: {audio_path.stat().st_size / (1024 * 1024):.2f} MB")
print(f"Preprocessed size: {preprocessed_audio_path.stat().st_size / (1024 * 1024):.2f} MB")

# Transcribe the compressed audio
result = transcribe_audio_file(preprocessed_audio_path)

print(json.dumps(result.metadata, indent=2, ensure_ascii=False))
print(result.text[:500])

Original file: /Users/nevena/Ironhack_Labs/week4/LAB11_FirstMatch_RightMatch/sources/The_Blueprint_For_Trustworthy_AI.m4a
Preprocessed file: /Users/nevena/Ironhack_Labs/week4/LAB11_FirstMatch_RightMatch/output/audio_preprocessed/The_Blueprint_For_Trustworthy_AI.mp3
Original size: 28.78 MB
Preprocessed size: 5.37 MB
{
  "source": "podcast",
  "filename": "The_Blueprint_For_Trustworthy_AI.mp3",
  "transcription_model": "whisper-1",
  "language": "english",
  "duration": 937.52001953125,
  "speaker": null
}
So, imagine for a second you're driving across, I don't know, a massive suspension bridge. You don't pull over halfway across, get out, and demand to see the blueprints, right? You don't interview the welding crew. No, you just trust it. You just drive. You trust the bridge. You trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's there. Exactly. But now, let's switch gears. Think ab

In [2]:
from pathlib import Path
import sys

repo_root = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src").exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.pdf_processor import extract_pdf_pages

eu_ai_act_path = repo_root / "sources" / "eu_ai_act.pdf"
trustworthy_ai_path = repo_root / "sources" / "altai_final_14072020_cs_accessible2_jsd5pdf_correct-title_3AC24743-DE11-0B7C-7C891D1484944E0A_68342.pdf"

eu_pages = extract_pdf_pages(eu_ai_act_path, source_id="eu_ai_act", document_type="eu_ai_act")
trust_pages = extract_pdf_pages(trustworthy_ai_path, source_id="trustworthy_ai", document_type="trustworthy_ai")

print(f"EU AI Act pages: {len(eu_pages)}")
print(f"Trustworthy AI pages: {len(trust_pages)}")
print(eu_pages[0].metadata)
print((eu_pages[0].text or trust_pages[0].text)[:300])


EU AI Act pages: 144
Trustworthy AI pages: 34
{'filename': 'eu_ai_act.pdf', 'source_id': 'eu_ai_act', 'document_type': 'eu_ai_act', 'page_number': 1, 'total_pages': 144}
REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL
of 13 June 2024
laying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directiv es 


In [3]:
print(eu_pages[20].metadata)
print(eu_pages[20].text[:300])

{'filename': 'eu_ai_act.pdf', 'source_id': 'eu_ai_act', 'document_type': 'eu_ai_act', 'page_number': 21, 'total_pages': 144}
(72) T o address concer ns relate d to opacity and complexity of cer tain AI systems and help deplo yers to fulfil their 
oblig ations under this Regulation, transparency should be required f or high-r isk AI syste ms bef ore they are placed 
on the market or put it into ser vice. High-r isk AI syst


In [4]:
print(trust_pages[10].metadata)
print(trust_pages[10].text[:300])

{'filename': 'altai_final_14072020_cs_accessible2_jsd5pdf_correct-title_3AC24743-DE11-0B7C-7C891D1484944E0A_68342.pdf', 'source_id': 'trustworthy_ai', 'document_type': 'trustworthy_ai', 'page_number': 11, 'total_pages': 34}
10 
General Safety 
• Did you define risks, risk metrics and risk levels of the AI system in each specific use 
case? 
o Did you put in place a process to continuously measure and assess risks? 
o Did you inform end-users and subjects of existing or potential risks? 
• Did you identify the possible 


In [5]:
from src.normalization import normalize_pdf_page, normalize_transcription

normalized_pdf = normalize_pdf_page(eu_pages[0])
normalized_transcript = normalize_transcription(result)

print(normalized_pdf)
print(normalized_transcript)

print(type(normalized_pdf).__name__, type(normalized_transcript).__name__)

print(sorted(normalized_pdf.__dataclass_fields__.keys()))
print(sorted(normalized_transcript.__dataclass_fields__.keys()))

print(normalized_pdf.document_id)
print(normalized_transcript.document_id)

NormalizedDocument(document_id='eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1', text='REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL\nof 13 June 2024\nlaying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, \n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and \nDirectiv es 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Ar tif icial Intelligence A ct)\n(T ext with EEA relevance)\nTHE EUR OPEAN P ARLIAMENT AND THE COUNCIL OF THE EUR OPEAN UNION,\nHaving regard to the T reaty on the Functioning of the European Union, and in par ticular Ar ticles 16 and 114 thereof,\nHaving regard to the proposal from the European Commission,\nAf ter transmission of the draf t legislative act to the national parliaments,\nHaving regard to the opinion of the European Economic and Social Committe e (\n1\n),\nHaving regard to the opinion of the European Central Bank (\n2\n),\nHaving regard to the opin

In [6]:
from src.chunking import chunk_document
from src.normalization import normalize_pdf_page, normalize_transcription

normalized_pdf = normalize_pdf_page(eu_pages[0])
normalized_transcript = normalize_transcription(result)

pdf_chunks = chunk_document(normalized_pdf)
transcript_chunks = chunk_document(normalized_transcript)

print(f"PDF chunks: {len(pdf_chunks)}")
print(f"Transcript chunks: {len(transcript_chunks)}")
print(pdf_chunks[0].text[:300])
print(pdf_chunks[0].metadata)
print([chunk.chunk_id for chunk in pdf_chunks])
print(transcript_chunks[0].text[:300])
print(transcript_chunks[0].metadata)
print([chunk.chunk_id for chunk in transcript_chunks])


PDF chunks: 5
Transcript chunks: 19
REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL
of 13 June 2024
laying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directiv es 
{'filename': 'eu_ai_act.pdf', 'source_id': 'eu_ai_act', 'document_type': 'eu_ai_act', 'page_number': 1, 'total_pages': 144, 'document_id': 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1', 'chunk_index': 0, 'chunk_start_char': 0, 'chunk_end_char': 1000, 'chunk_size': 1000, 'chunk_overlap': 100}
['eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0000', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0001', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0002', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0003', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0004']
So, imagine for a second you're driving across, I don't know, a massive suspension bridge. You do

In [7]:
from src.chunking import chunk_documents
from src.embeddings import DEFAULT_EMBEDDING_MODEL, embed_chunks
from src.normalization import normalize_pdf_page, normalize_transcription

normalized_pdf = normalize_pdf_page(eu_pages[0])
normalized_transcript = normalize_transcription(result)

chunks = chunk_documents([normalized_pdf, normalized_transcript])
embedded_chunks = embed_chunks(chunks)
first_chunk = embedded_chunks[0]

print(f"Total embedded chunks: {len(embedded_chunks)}")
print(f"Embedding model used: {DEFAULT_EMBEDDING_MODEL}")
print(f"Embedding vector length: {len(first_chunk.embedding)}")
print(f"First five embedding values: {first_chunk.embedding[:5]}")
print(f"First chunk ID: {first_chunk.chunk_id}")
print(f"First document ID: {first_chunk.document_id}")
print(f"First chunk metadata: {first_chunk.metadata}")
print(all(len(item.embedding) == len(first_chunk.embedding) for item in embedded_chunks))


Total embedded chunks: 24
Embedding model used: text-embedding-3-small
Embedding vector length: 1536
First five embedding values: [0.0211181640625, 0.01444244384765625, 0.01279449462890625, -0.0029315948486328125, -0.001064300537109375]
First chunk ID: eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0000
First document ID: eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1
First chunk metadata: {'filename': 'eu_ai_act.pdf', 'source_id': 'eu_ai_act', 'document_type': 'eu_ai_act', 'page_number': 1, 'total_pages': 144, 'document_id': 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1', 'chunk_index': 0, 'chunk_start_char': 0, 'chunk_end_char': 1000, 'chunk_size': 1000, 'chunk_overlap': 100}
True


In [8]:
from pinecone import Pinecone, ServerlessSpec

In [9]:
# index, cloud and region created from here
'''
from dotenv import load_dotenv
import os
from pinecone import Pinecone, ServerlessSpec

# Load variables from .env
load_dotenv()

# Get Pinecone API key
api_key = os.getenv("PINECONE_API_KEY")

if not api_key:
    raise ValueError("PINECONE_API_KEY not found in .env file")

# Initialize Pinecone
pc = Pinecone(api_key=api_key)

index_name = "ironhack-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

print(f"Index '{index_name}' is ready.")
'''

'\nfrom dotenv import load_dotenv\nimport os\nfrom pinecone import Pinecone, ServerlessSpec\n\n# Load variables from .env\nload_dotenv()\n\n# Get Pinecone API key\napi_key = os.getenv("PINECONE_API_KEY")\n\nif not api_key:\n    raise ValueError("PINECONE_API_KEY not found in .env file")\n\n# Initialize Pinecone\npc = Pinecone(api_key=api_key)\n\nindex_name = "ironhack-rag"\n\nif not pc.has_index(index_name):\n    pc.create_index(\n        name=index_name,\n        dimension=1536,\n        metric="cosine",\n        spec=ServerlessSpec(\n            cloud="aws",\n            region="us-east-1"\n        )\n    )\n\nprint(f"Index \'{index_name}\' is ready.")\n'

In [10]:
from dotenv import load_dotenv
load_dotenv()

import os

print(os.getenv("PINECONE_INDEX_NAME"))
print(os.getenv("PINECONE_CLOUD"))
print(os.getenv("PINECONE_REGION"))

ironhack-rag
AWS
us-east-1


In [11]:
import inspect
from src import vector_store

print(inspect.getsource(vector_store.upsert_chunks))

def upsert_chunks(
    chunks: list[EmbeddedChunk],
    *,
    index_name: str | None = None,
    namespace: str = DEFAULT_NAMESPACE,
) -> Any:
    """Upsert a list of embedded chunks into Pinecone."""

    if not chunks:
        return None

    index = get_index(index_name=index_name)
    vectors = [_as_vector_record(chunk) for chunk in chunks]

    try:
        return index.upsert(vectors=vectors, namespace=namespace)
    except Exception as exc:  # noqa: BLE001 - keep network/config failures readable
        raise VectorStoreError(f"Failed to upsert chunks into Pinecone: {exc}") from exc



In [12]:
from src import vector_store
import inspect

print(vector_store.__file__)
print(inspect.getsource(vector_store.upsert_chunks))

/Users/nevena/Ironhack_Labs/week4/LAB11_FirstMatch_RightMatch/src/vector_store.py
def upsert_chunks(
    chunks: list[EmbeddedChunk],
    *,
    index_name: str | None = None,
    namespace: str = DEFAULT_NAMESPACE,
) -> Any:
    """Upsert a list of embedded chunks into Pinecone."""

    if not chunks:
        return None

    index = get_index(index_name=index_name)
    vectors = [_as_vector_record(chunk) for chunk in chunks]

    try:
        return index.upsert(vectors=vectors, namespace=namespace)
    except Exception as exc:  # noqa: BLE001 - keep network/config failures readable
        raise VectorStoreError(f"Failed to upsert chunks into Pinecone: {exc}") from exc



In [13]:
import time

from src.chunking import chunk_documents
from src.embeddings import embed_chunks
from src.normalization import normalize_pdf_page, normalize_transcription
from src.vector_store import create_index_if_needed, get_index, query_index, upsert_chunks

normalized_pdf = normalize_pdf_page(eu_pages[0])
normalized_transcript = normalize_transcription(result)
chunks = chunk_documents([normalized_pdf, normalized_transcript])
embedded_chunks = embed_chunks(chunks)

index_name = create_index_if_needed(dimension=len(embedded_chunks[0].embedding))
index = get_index(index_name)
upsert_response = upsert_chunks(embedded_chunks, index_name=index_name)
time.sleep(10)
query_response = query_index(embedded_chunks[0].embedding, top_k=3, index_name=index_name)
matches = query_response.matches

print(f"Index name: {index_name}")
print(f"Uploaded vectors: {len(embedded_chunks)}")
print(f"Upsert response: {upsert_response}")
print([match.id for match in matches])
print([match.score for match in matches])
print([match.metadata.get('document_id') for match in matches])
print([match.metadata.get('source_id') for match in matches])
print(matches[0].metadata.get('text', '')[:200])


Index name: ironhack-rag
Uploaded vectors: 24
Upsert response: UpsertResponse(upserted_count=24)
['eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0000', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0003', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1:chunk-0001']
[1.00035727, 0.714134574, 0.695881307]
['eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1', 'eu_ai_act:eu_ai_act:eu_ai_act.pdf:page-1']
['eu_ai_act', 'eu_ai_act', 'eu_ai_act']
REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL
of 13 June 2024
laying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, 
(E
